# Module 1.3 — MCP: Standardizing Tool Discovery

**Where we are.** Notebook 2 gave the agent a `calculator` and a `unit_converter`. Notebook 3 added `search_papers`. Every one of those was a hand-crafted Python function we decorated with `@tool`. The tool lived in the *same process* as the agent, and changing the tool meant changing the notebook.

**The problem this creates at scale.** Imagine you want your agent to use ten tools from one team, twenty from another, and a local-only simulator you wrote yourself. Hardcoding every tool into every agent is a combinatorial mess. Worse, when someone fixes a bug in the arXiv search tool, every agent that uses it has to update.

**The fix — a protocol.** The **Model Context Protocol** (MCP) is a JSON-RPC specification that separates *who owns the tool* from *who uses the tool*:

- An **MCP server** advertises a catalogue of tools. Each tool has a name, a description, and a JSON Schema for its arguments.
- An **MCP client** (the agent) connects to the server, asks `tools/list`, and calls `tools/call(name, args)` with whatever the user wants.
- Transport is either **stdio** (one subprocess, for local tools) or **HTTP+SSE** (for remote tools).

MCP is deliberately boring. It's the USB-C of AI agents: a universal adapter so anyone can publish tools, anyone can consume them, and neither side has to know about the other.

**Physicist's bridge.** Compare this with how a control system at a synchrotron talks to beamline instruments: each instrument exposes a *device schema*, the control system queries it at runtime, and the experiment code doesn't care whether the detector is a CCD or a silicon drift — it just calls `detector.acquire(time_s=1.0)`. MCP is the same idea for LLM tools.

**Plan.**

- **Part A (~20 min)** — Talk to the MCP server by hand, over stdio, in Python. You'll see the actual JSON-RPC messages for `tools/list` and `tools/call`. This is the "know what's underneath" part.
- **Part B (~25 min)** — Plug the same server into `smolagents` via `ToolCollection.from_mcp()`. Two demo tasks: a live arXiv search and a phase-transition tour with the real Monte Carlo code from Module 1.0.

**What changed since Module 1.2.** Two runtime choices, both driven by hands-on experience with small local models:

1. **`CodeAgent` instead of `ToolCallingAgent`.** Small Llamas (7–8 B) are much better at writing a short Python snippet than at emitting strictly-valid JSON function calls. `CodeAgent` executes model-generated Python inside a sandboxed interpreter that has the tools exposed as ordinary Python names. That format is native to these models — tool use becomes code.
2. **Default model is `qwen2.5:7b`.** Community experience (and ours with Module 1.2) shows it handles small-model tool use more reliably than `llama3.1:8b`. Avoid "thinking"/reasoning-token models here — the extra `<think>…</think>` chatter confuses framework parsers and blows the context budget.


## 1. The server we already have — `mcp_server/physics_tools_server.py`

In Phase 1 we built an MCP server with two tools:

| Tool | What it does |
|---|---|
| `search_arxiv(query, max_results)` | Live query to the arXiv API; returns title/authors/abstract/id |
| `run_ising_simulation(lattice_size, temperature, num_steps, algorithm, ...)` | Runs real Monte Carlo via `ising_simulator.py`; returns magnetisation, energy, susceptibility, specific heat |

Let's inspect the server's source. The `@mcp.tool()` decorator turns each Python function into an MCP-advertised tool; FastMCP auto-generates the JSON Schema from the type hints.


In [1]:
from pathlib import Path
src = Path("../mcp_server/physics_tools_server.py").read_text()
# Show just the two @mcp.tool() blocks so students see the contract.
for chunk in src.split("@mcp.tool()"):
    if "def search_arxiv" in chunk or "def run_ising_simulation" in chunk:
        # keep first ~25 lines of each tool
        print("@mcp.tool()" + "\n".join(chunk.splitlines()[:25]))
        print("\n" + "-" * 72 + "\n")


@mcp.tool()
def search_arxiv(query: str, max_results: int = 5) -> dict[str, Any]:
    """Search the arXiv preprint server and return the top results.

    Args:
        query: free-text search query (title/abstract/authors), e.g.
            "2D Ising model Monte Carlo" or "Wolff cluster algorithm".
            For better precision use arXiv's field prefixes and quoted
            phrases, e.g. ``ti:"Ising" AND abs:"neural network"``.
        max_results: maximum number of hits to return (1..20). Defaults to 5.

    Returns:
        A dict of the form::

            {
              "query": "...",
              "count": N,
              "results": [
                 {"title": ..., "authors": [...], "abstract": ...,
                  "arxiv_id": ..., "url": ..., "published": "YYYY-MM-DD"},
                 ...
              ]
            }

        On transient failure (e.g. arXiv rate-limiting, HTTP 429), the

------------------------------------------------------------------------

@m

## 2. Part A — spin up the server and ask `tools/list`

MCP's stdio transport is the simplest to run: we launch `python -m mcp_server.physics_tools_server` as a subprocess, its stdin/stdout become the JSON-RPC pipe, and the `mcp` Python SDK gives us a client.

The first thing a client does is `initialize` (protocol handshake), then `tools/list` (discover what's on offer). The tool schemas it gets back are exactly what an LLM needs to decide *whether* to call each tool.


In [2]:
import asyncio
import os
import sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# How to start the server. We invoke the module from the project root,
# so the import path sees `mcp_server.*`.
SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,
    args=["-m", "mcp_server.physics_tools_server"],
    env={**os.environ, "PYTHONPATH": ".."},
    cwd="..",
)


async def list_tools_once() -> None:
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            resp = await session.list_tools()
            for t in resp.tools:
                print(f"- {t.name}")
                print(f"    desc: {(t.description or '').splitlines()[0][:80]}...")
                props = t.inputSchema.get("properties", {})
                for pname, pspec in props.items():
                    ptype = pspec.get("type", pspec.get("anyOf", "?"))
                    default = f"  (default: {pspec['default']})" if "default" in pspec else ""
                    print(f"      {pname}: {ptype}{default}")


# In Jupyter, an event loop is already running; use await directly.
await list_tools_once()


- search_arxiv
    desc: Search the arXiv preprint server and return the top results....
      query: string
      max_results: integer  (default: 5)
- run_ising_simulation
    desc: Run a 2D Ising-model Monte Carlo simulation and return observables....
      lattice_size: integer
      temperature: number
      num_steps: integer
      algorithm: string  (default: wolff)
      thermalization_steps: [{'type': 'integer'}, {'type': 'null'}]  (default: None)
      seed: [{'type': 'integer'}, {'type': 'null'}]  (default: None)


## 3. Part A continued — call `run_ising_simulation` by hand

Now the interesting call: `tools/call`. We give the tool's name and a dict of arguments; the server returns content blocks. FastMCP serialises structured returns as JSON-encoded text, which is what we'll see below.

This is genuinely the same wire as a production MCP deployment. Everything else in "agent frameworks" is sugar on top of these two RPCs.


In [3]:
import json as _json
import time


async def call_ising_once(L: int, T: float, n: int) -> dict:
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            t0 = time.time()
            result = await session.call_tool(
                "run_ising_simulation",
                {"lattice_size": L, "temperature": T, "num_steps": n,
                 "algorithm": "wolff", "seed": 42},
            )
            dt = time.time() - t0
            # FastMCP returns a list of content blocks. For a dict-returning
            # tool, block 0 is a TextContent carrying the JSON string.
            payload = _json.loads(result.content[0].text)
            payload["_elapsed_s"] = dt
            return payload


run = await call_ising_once(L=16, T=2.269, n=2000)
for k, v in run.items():
    if isinstance(v, float):
        print(f"  {k:20s}: {v:+.4f}")
    elif isinstance(v, (int, str)):
        print(f"  {k:20s}: {v}")


  lattice_size        : 16
  temperature         : +2.2690
  num_steps           : 2000
  thermalization_steps: 400
  algorithm           : wolff
  magnetization_mean  : +0.7319
  magnetization_std   : +0.1725
  energy_mean         : -1.4709
  energy_std          : +0.1637
  specific_heat       : +1.3326
  susceptibility      : +3.3580
  mean_cluster_size   : +142.8710
  elapsed_seconds     : +0.4178
  _elapsed_s          : +0.4524


### What we just did, in one breath

We launched a subprocess, sent it two JSON-RPC messages, parsed the reply. That is MCP. A 32-line Python file. The pieces to remember:

- The server is **discoverable**: `tools/list` returns machine-readable schemas. An agent that has never seen this server before knows how to use it from the first reply.
- The server is **stateless** per call: each `tools/call` is self-contained, so scaling or swapping servers doesn't break clients.
- The server is **language-agnostic**: ours is Python, but the agent wouldn't know or care if it were a Rust or TypeScript server. That's the composability win.

Contrast with Modules 1.1 and 1.2: there, the tool was a Python function living inside our notebook. That's fine for teaching, but means the agent and the tool share a process, a virtual-env, and an author. MCP splits that apart.


## 5. Part B — bridge to `smolagents` with `ToolCollection.from_mcp`

`smolagents` has first-class MCP support through a tiny adapter library (`mcpadapt`). One call, `ToolCollection.from_mcp(server_parameters)`, launches the server, discovers its tools, and hands them back as native smolagents `Tool` objects we can drop straight into a `CodeAgent`.

One subtlety for a notebook: `ToolCollection.from_mcp()` is a **context manager** — the server only lives inside a `with` block. That's awkward for cell-by-cell exploration, so we enter the context manager *manually* and tear it down at the end of the notebook. (You could also write one giant cell with `with ... as tc:`; both patterns are valid.)


In [4]:
from smolagents import ToolCollection, CodeAgent, LiteLLMModel

# Manually enter the context so later cells can re-use `tc`.
_mcp_cm = ToolCollection.from_mcp(SERVER_PARAMS)
tc = _mcp_cm.__enter__()

print("MCP tools available to smolagents:")
for t in tc.tools:
    print(f"  - {t.name}")


MCP tools available to smolagents:
  - search_arxiv
  - run_ising_simulation


## 6. Build a `CodeAgent`

Two changes from notebook 02:

1. **`CodeAgent` instead of `ToolCallingAgent`.** The model will emit Python code, which we run inside a sandboxed interpreter. The tools are in scope as Python callables — e.g. the model types `run_ising_simulation(lattice_size=32, temperature=2.269, num_steps=10000)` and that really runs.
2. **`additional_authorized_imports`** — the sandbox is by default very restricted. We enable a narrow whitelist: `math`, `json` (because MCP tools return JSON strings we may want to parse), `numpy` (useful if the agent wants to manipulate returned arrays).

Default model is `qwen2.5:7b`. If you don't have it pulled, `ollama pull qwen2.5:7b` (~4.7 GB) or swap for `llama3.1:8b`. Larger models improve reliability; the pattern is identical.


In [5]:
MODEL = "qwen2.5:7b"  # swap to "llama3.1:8b" or any chat model you have

model = LiteLLMModel(
    model_id=f"ollama_chat/{MODEL}",
    api_base="http://localhost:11434",
    api_key="ollama",
    temperature=0.0,
)

agent = CodeAgent(
    tools=[*tc.tools],
    model=model,
    additional_authorized_imports=["math", "json", "numpy"],
    max_steps=6,
)

print("CodeAgent ready. Tools:", list(agent.tools.keys()))


CodeAgent ready. Tools: ['search_arxiv', 'run_ising_simulation', 'final_answer']


## 7. Demo 1 — live arXiv search

Ask the agent to find recent work on machine learning approaches to the Ising model and summarise the top three hits. Expect 1–2 tool calls: one `search_arxiv(query="...", max_results=3)`, then a code block that iterates over the returned papers to build a summary.

The tool's underlying Python return is a dict of the shape

```python
{
  "query": "...",
  "count": 3,
  "results": [
    {"title": ..., "authors": [...], "abstract": ...,
     "arxiv_id": ..., "url": ..., "published": "YYYY-MM-DD"},
    ...
  ]
}
```

…but **MCP transports it as a JSON-encoded string**. When the `CodeAgent` calls the tool, what it receives is a `str`, and it must `json.loads(...)` the string before it can index into the `"results"` key. That is why we added `json` to `additional_authorized_imports` in §6. You'll see the same pattern in §8 with `run_ising_simulation`.

> **Why a dict and not a list?** FastMCP serialises a Python list return as *multiple* content blocks (one per element), and smolagents' `mcpadapt` bridge only reads the first block — which would silently hand the agent a single paper no matter what `max_results` we asked for. Wrapping the list inside a top-level dict forces FastMCP to send one content block carrying the whole payload. *If your agent ever looks like it's only getting one result from a list-returning MCP tool, this is the bug to check for first.*

> **What counts as a good run.** You want the agent to (a) call the tool once, (b) parse the JSON string, (c) iterate over the `"results"` list, (d) not re-invent abstracts from memory, and (e) mention the arXiv IDs. If it skips the tool and makes up papers, that's the exact failure mode MCP is designed to prevent — the tool is right there, let it do its job. If it calls the tool but forgets `json.loads`, you'll see a `"subscript a string with a string index"` error — that's your cue that the agent needs a stronger hint in the task prompt.

> **Heads-up — arXiv rate limits.** The public arXiv API throttles aggressively: roughly **1 request per 3 s per IP**, and once you trip the limit the block can persist for **15–30 minutes**. If you re-run this cell too eagerly (or a classmate behind the same NAT did), you'll see `HTTP 429` errors. The server in `mcp_server/physics_tools_server.py` handles this two ways:
>
> 1. **Internal retry with backoff** — up to 3 attempts, 10 s → 20 s between tries, with a baseline 5 s spacing between page requests.
> 2. **Graceful error return** — if all retries still fail, `search_arxiv` returns the same dict shape above but with `results == []` and two extra keys, `"error"` and `"hint"`, **instead of raising**. Because we're using `CodeAgent`, the model sees that as ordinary Python data and can reason about it — typically by rephrasing the query, apologising to the user, or (if `search_papers` from Module 1.2 is also loaded) falling back to the local RAG corpus.
>
> This is a small but important agent-design pattern: **make transient tool failures data, not exceptions**, so the LLM can handle them with the same mechanism it uses for any other tool output. An exception terminates the step; a dict with an `"error"` key becomes context the model can actually reason about.

> **Query-quality caveat.** The LLM may phrase its query as something like `"machine-learning AND 2D Ising model"`. arXiv's free-text scorer treats `AND` as a literal word and hyphenation quirkily, so you can get off-topic hits like a fake-news-detection survey ranking high. For sharper results use arXiv's field prefixes: e.g. `ti:"Ising" AND abs:"neural network"`. We leave the query to the agent on purpose — it's informative to watch what a small model picks.


In [6]:
task1 = (
    "Search arXiv for recent papers on machine-learning approaches to the 2D "
    "Ising model (e.g. neural networks for phase classification, "
    "variational Monte Carlo with ML, generative models for spin "
    "configurations). The `search_arxiv` tool returns a JSON-encoded "
    "string; parse it with `json.loads` to obtain a dict whose "
    "'results' key holds the list of papers. Summarise the top 3 hits: "
    "one sentence per paper, with its arXiv id."
)
print(agent.run(task1))


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search arXiv for recent papers on machine-learning approaches to the 2D Ising model (e.g. neural networks for   │
│ phase classification, variational Monte Carlo with ML, generative models for spin configurations). The          │
│ `search_arxiv` tool returns a JSON-encoded string; parse it with `json.loads` to obtain a dict whose 'results'  │
│ key holds the list of papers. Summarise the top 3 hits: one sentence per paper, with its arXiv id.              │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5:7b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/Users/Yak52/coding-lab/ising-agents-course/.venv/lib/python3.10/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 5 fields but got 4: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content='Thought:...one, function_call=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ne, function_call=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import json                                                                                                      
                                                                                                                   
  results = search_arxiv(query="machine-learning AND 2D Ising model", max_results=10)                              
  parsed_results = json.loads(results)                                                                             
                                                                                                                   
  # Extract the titles and arXiv IDs of the top 3 papers                                                           
  top_papers = parsed_results["results"][:3]                                                                       
  summary = []                                                                                                     
                                                                                                                   
  for paper in top_papers:                                                                                         
      summary.append(f"{paper['title']} - {paper['arxiv_id']}")                                                    
                                                                                                                   
  final_answer("\n".join(summary))                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: Active learning for data streams: a survey - 2302.08893v4
Emotion in Reinforcement Learning Agents and Robots: A Survey - 1705.05172v1
Learning Curves for Decision Making in Supervised Machine Learning: A Survey - 2201.12150v2

[Step 1: Duration 35.17 seconds| Input tokens: 3,235 | Output tokens: 158]

Active learning for data streams: a survey - 2302.08893v4
Emotion in Reinforcement Learning Agents and Robots: A Survey - 1705.05172v1
Learning Curves for Decision Making in Supervised Machine Learning: A Survey - 2201.12150v2


## 8. Demo 2 — the agent runs the phase transition

This is the SA Part's centrepiece. We ask the agent to run three simulations spanning the phase transition and describe what it sees. Each `run_ising_simulation` call is a real Monte Carlo run on a 32×32 lattice — the same code you built in Phase 1.

Expect $|M|$ to drop from near 1 (ordered, $T < T_c$) through $\sim 0.5$ with visibly larger fluctuations (critical, $T = T_c$) down to near 0 (disordered, $T > T_c$). The magic is that *none* of these numbers are in the LLM's training data — they are generated right now, in this kernel, by your simulator.

> If the agent rambles or fails on JSON parsing, remember: the tool returns a **JSON string**, not a Python dict. We imported `json` in the sandbox for exactly this reason — the agent should use `json.loads(...)` to pull out fields.


In [7]:
task2 = (
    "Run a 2D Ising Monte Carlo simulation on a 32x32 lattice with the "
    "Wolff algorithm and 10000 steps at each of the temperatures T = 2.0, "
    "T = 2.269 (Onsager's T_c), and T = 3.0. For each run, parse the "
    "returned JSON and extract magnetization_mean, magnetization_std, and "
    "susceptibility. Then describe in plain English what happens to the "
    "magnetization across these three temperatures, and why."
)
print(agent.run(task2))


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Run a 2D Ising Monte Carlo simulation on a 32x32 lattice with the Wolff algorithm and 10000 steps at each of    │
│ the temperatures T = 2.0, T = 2.269 (Onsager's T_c), and T = 3.0. For each run, parse the returned JSON and     │
│ extract magnetization_mean, magnetization_std, and susceptibility. Then describe in plain English what happens  │
│ to the magnetization across these three temperatures, and why.                                                  │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5:7b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/Users/Yak52/coding-lab/ising-agents-course/.venv/lib/python3.10/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 5 fields but got 4: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content="Thought:...one, function_call=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ne, function_call=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  results = []                                                                                                     
  temperatures = [2.0, 2.269, 3.0]                                                                                 
  for temp in temperatures:                                                                                        
      result = run_ising_simulation(lattice_size=32, temperature=temp, num_steps=10000, algorithm='wolff')         
      results.append(result)                                                                                       
                                                                                                                   
  final_answer(results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: ['{\n  "lattice_size": 32,\n  "temperature": 2.0,\n  "num_steps": 10000,\n  
"thermalization_steps": 2000,\n  "algorithm": "wolff",\n  "magnetization_mean": 0.9107841796875,\n  
"magnetization_std": 0.027303844887498242,\n  "energy_mean": -1.744489453125,\n  "energy_std": 
0.05322930728367735,\n  "specific_heat": 0.7253399433984374,\n  "susceptibility": 0.3816959721679609,\n  
"acceptance_rate": null,\n  "mean_cluster_size": 851.5148,\n  "elapsed_seconds": 10.29063883330673,\n  
"final_configuration": [\n    [\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n     
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      1,\n      -1,\n      -1,\n      -1,\n 
-1,\n      -1,\n      -1\n    ],\n    [\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n 
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      1,\n      -1,\n      -1,\n      -1,\n 
1,\n      -1,\n      -1,\n      -1\n    ],\n    [\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      1,\n      -1,\n      -1,\n 
-1,\n      -1,\n      -1,\n      -1,\n      -1\n    ],\n    [\n      -1,\n      -1,\n      -1,\n      -1,\n      
1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n 
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      1,\n      -1,\n 
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1\n    ],\n    [\n      -1,\n      -1,\n      -1,\n      
1,\n      1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n  
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1\n    ],\n    [\n      -1,\n      -1,\n      
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      1,\n      -1\n    ],\n    [\n      -1,\n      
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      1,\n      -1\n    ],\n    [\n      
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1\n    ],\n    
[\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n  
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1\n 
],\n    [\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n     
-1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n
-1,\n      -1,\n      -1,\n      

[Step 1: Duration 56.69 seconds| Input tokens: 3,243 | Output tokens: 138]

['{\n  "lattice_size": 32,\n  "temperature": 2.0,\n  "num_steps": 10000,\n  "thermalization_steps": 2000,\n  "algorithm": "wolff",\n  "magnetization_mean": 0.9107841796875,\n  "magnetization_std": 0.027303844887498242,\n  "energy_mean": -1.744489453125,\n  "energy_std": 0.05322930728367735,\n  "specific_heat": 0.7253399433984374,\n  "susceptibility": 0.3816959721679609,\n  "acceptance_rate": null,\n  "mean_cluster_size": 851.5148,\n  "elapsed_seconds": 10.29063883330673,\n  "final_configuration": [\n    [\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1\n    ],\n    [\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n      -1,\n  

## 9. Clean shutdown

Because we entered the context manager manually, we should exit it cleanly at the end of the notebook. This terminates the server subprocess and closes the stdio pipes. Forgetting this leaves a zombie Python process around — annoying but harmless.


In [ ]:
_mcp_cm.__exit__(None, None, None)
print("MCP server stopped.")


## 10. Key takeaway + exercises

**The shape of the protocol.** MCP is two RPCs (`tools/list`, `tools/call`), a schema (JSON Schema), and a transport (stdio or HTTP+SSE). That's all. Everything else — the ecosystem of community servers, the `smolagents` adapter, the IDE integrations — builds on those two calls.

**Why this matters pedagogically.** You now have four tool-delivery mechanisms on the shelf:

| Mechanism | Example | When to use |
|---|---|---|
| Hand-wired ReAct | `calculator` in notebook 2 | Teaching, debugging, absolute control |
| `smolagents @tool` | `search_papers` in notebook 3 | Single-repo agents, in-process |
| MCP server (yours) | `physics_tools_server.py` | Cross-agent reuse, multi-language, remote |
| MCP server (someone else's) | `pubmedmcp`, `slack-mcp`, ... | Leverage community tools — zero code |

**Why `CodeAgent` over `ToolCallingAgent` for small models.** When the LLM has to choose a tool and supply arguments, "write a Python line" is a distribution much closer to the model's pretraining than "emit structured JSON matching this schema." Frontier models do either equally well; small local models (7–8 B) write Python noticeably more reliably.

### Exercises

1. **Add a tool to the MCP server.** Pick one Ising observable not yet exposed — e.g. `compute_binder_cumulant(lattice_size, temperature, num_steps)` returning $U_4 = 1 - \langle m^4 \rangle / (3 \langle m^2 \rangle^2)$, or `compute_correlation_length` via a fit to the two-point function. Add it to `mcp_server/physics_tools_server.py`, restart the agent, and ask a question that needs it. Notice: **you did not touch the notebook or the agent code.** That's MCP's decoupling doing its job.

2. **Swap the model.** Re-run §7 and §8 with `MODEL = "llama3.1:8b"`. Which tasks degrade? If you run the arXiv search twice, do the agents even pick the same papers? Small fluctuations in the sampling distribution cascade into very different tool-call chains — a reliability axis worth measuring before you trust any agent in production.

3. **Three phase points in a finite-size scan.** Modify task 2 to also sweep the lattice size: ask for $L \in \{16, 32, 64\}$ at $T = T_c$. The simulator should give $|M| \sim L^{-\beta/\nu}$ with $\beta/\nu = 1/8$ in 2D — a tiny finite-size-scaling check. If the agent gets the exponent wrong or doesn't attempt a fit, that's a sign your prompt needs to be more specific about *what* to compute. This exercise foreshadows MAS Part, where one specialised agent *is* a numerical analyst.

4. **Combine with Module 1.2's tools.** `[*tc.tools]` is just a list — nothing stops you from concatenating it with `calculator_tool`, `search_papers_tool`, etc. Build an agent with all five tools and ask it to (i) search arXiv for papers on the 3D Ising model critical exponents, (ii) simulate the 2D one at a matching temperature, and (iii) reconcile — with citations. This is the Module-1 climax.

---

**Next up — Module 1.4:** The agent currently *acts* confidently even when it is wrong. We'll wrap it in a **Critic** that reads its output, compares simulation numbers to Onsager's exact result, and asks for a revision if something looks off. That's *reflection*. Then we give it a memory of past mistakes (*reflexion*, with an x) that persists across sessions.
